In [88]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# **Upload Raw Dataset**

In [89]:
team_matches = pd.read_csv("/content/Team_Matches_Raw.csv")

# **Team_matches Overview**

In [90]:
# shape of the Dataset
print("Players Info Shape:", team_matches.shape)
print("Rows:", team_matches.shape[0])
print("Columns:", team_matches.shape[1])

Players Info Shape: (15808, 19)
Rows: 15808
Columns: 19


In [91]:
team_matches.head()

,id,team_name,round,match_date,year,home_away,opponent,team_quarter_scores,team_score,opponent_quarter_scores,opponent_score,result,margin,venue,crowd,team_goals_kicked,team_behinds,opponent_goals_kicked,opponent_behinds
0,15807,Hawthorn Hawks,QF,1994-09-10,1994,A,North Melbourne Kangaroos,4.5 5.7 10.11 13.13 13.13 13.13,91,2.3 6.12 9.12 12.19 13.23 15.24,114,L,23,Waverley Park,38223.0,13,13,15,24
1,15808,North Melbourne Kangaroos,QF,1994-09-10,1994,H,Hawthorn Hawks,2.3 6.12 9.12 12.19 13.23 15.24,114,4.5 5.7 10.11 13.13 13.13 13.13,91,W,6,Waverley Park,38223.0,15,24,13,13
2,5646,North Melbourne Kangaroos,10,2008-05-31,2008,A,Brisbane Lions,2.2 6.2 12.3 15.8,98,4.7 11.12 15.17 18.21,129,L,-31,The Gabba,22118.0,15,8,18,21
3,8829,Sydney Swans,15,2017-06-30,2017,A,Melbourne Demons,1.8 5.15 8.16 11.19,85,4.0 4.1 5.4 7.8,50,W,35,Melbourne Cricket Ground,47464.0,11,19,7,8
4,8873,Sydney Swans,11,2019-06-01,2019,A,Geelong Cats,3.3 5.8 6.12 8.15,63,5.1 7.2 11.4 13.7,85,L,-22,GMHBA Stadium,29021.0,8,15,13,7


In [92]:
team_matches.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15808 entries, 0 to 15807
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       15808 non-null  int64  
 1   team_name                15808 non-null  object 
 2   round                    15808 non-null  object 
 3   match_date               15808 non-null  object 
 4   year                     15808 non-null  int64  
 5   home_away                15808 non-null  object 
 6   opponent                 15808 non-null  object 
 7   team_quarter_scores      15808 non-null  object 
 8   team_score               15808 non-null  int64  
 9   opponent_quarter_scores  15808 non-null  object 
 10  opponent_score           15808 non-null  int64  
 11  result                   15808 non-null  object 
 12  margin                   15808 non-null  int64  
 13  venue                    15808 non-null  object 
 14  crowd                 

**Missing Values and Duplicates**

In [93]:
team_matches.isnull().sum()

,0
id,0
team_name,0
round,0
match_date,0
year,0
home_away,0
opponent,0
team_quarter_scores,0
team_score,0
opponent_quarter_scores,0


In [94]:
print("Duplicate rows:", team_matches.duplicated().sum())

Duplicate rows: 0


# **Team_Matches Cleaning**

In [95]:
# Check data types
print("Before Cleaning:")
print(team_matches.dtypes)

# Fix date data type
team_matches["match_date"] = pd.to_datetime(team_matches["match_date"],errors="coerce")

# Fix crowd data type
team_matches["crowd"] = pd.to_numeric(team_matches["crowd"],errors="coerce")

# Remove duplicates
team_matches.drop_duplicates()

# Calculate median
median_crowd = team_matches["crowd"].median()
print("\nMedian Crowd:", median_crowd)

# Fill missing values with median
team_matches["crowd"] = team_matches["crowd"].fillna(median_crowd)
team_matches["crowd"] = team_matches["crowd"].astype("Int64")

# Check after cleaning
print("\nAfter Cleaning:")
print(team_matches.dtypes)

# 7. Check missing crowd values
print("\nMissing Crowd Values:")
print(team_matches["crowd"].isnull().sum())

Before Cleaning:
id                           int64
team_name                   object
round                       object
match_date                  object
year                         int64
home_away                   object
opponent                    object
team_quarter_scores         object
team_score                   int64
opponent_quarter_scores     object
opponent_score               int64
result                      object
margin                       int64
venue                       object
crowd                      float64
team_goals_kicked            int64
team_behinds                 int64
opponent_goals_kicked        int64
opponent_behinds             int64
dtype: object

Median Crowd: 29814.0

After Cleaning:
id                                  int64
team_name                          object
round                              object
match_date                 datetime64[ns]
year                                int64
home_away                          object
opponent    

In [96]:
# Final validation
print("\nFinal Shape:", team_matches.shape)
print("Duplicate Rows:", team_matches.duplicated().sum())
print("Missing Values:", team_matches.isnull().sum().sum())


Final Shape: (15808, 19)
Duplicate Rows: 0
Missing Values: 0


# **Saved Clean Dataset into CSV Format**

In [97]:
team_matches.to_csv("Team_Matches_Cleaned.csv", index=False)
print("Cleaning completed!")

Cleaning completed!


# **Actual Implementation Task**

In [98]:
# Load datasets
players = pd.read_csv("/content/Players_round_by_round_stat_cleaned.csv")
matches = pd.read_csv("/content/Team_Matches_Cleaned.csv")

# **Task 1 : Relationship Discovery**

In [99]:
# Find common columns
common_columns = players.columns.intersection(matches.columns)
print("Common columns:")
print(list(common_columns))

# Make team names consistent
players["team"] = players["team"].str.strip()
matches["team_name"] = matches["team_name"].str.strip()
matches["team_name"] = matches["team_name"].replace("W. Bulldogs", "Western Bulldogs")

# Rename team_name so both datasets have the same column name
matches = matches.rename(columns={"team_name": "team"})

# Check whether the possible merge key is unique in match data
merge_key = ["team", "round", "year", "match_date"]


Common columns:
['id', 'year', 'opponent', 'round', 'result', 'match_date', 'margin']


In [100]:
duplicate_keys = matches.duplicated(merge_key).sum()

print("\nMerge key:")
print(merge_key)

print("\nDuplicate merge keys in Team Match dataset:")
print(duplicate_keys)

print("\nConclusion:")
print("A composite key is required because team, round, year and match_date together identify the specific team match.")


Merge key:
['team', 'round', 'year', 'match_date']

Duplicate merge keys in Team Match dataset:
0

Conclusion:
A composite key is required because team, round, year and match_date together identify the specific team match.


# **Task 2 : Context Enrichment**

In [102]:
# Clean the team names
players["team"] = players["team"].astype(str).str.strip()

matches["team"] = matches["team"].astype(str).str.strip()

# Make team names the same in both datasets
matches["team"] = matches["team"].replace("W. Bulldogs","Western Bulldogs")

# Rename team_name to team
matches = matches.rename(columns={"team_name": "team"})

# Make merge columns the same data type
players["team"] = players["team"].astype(str)
matches["team"] = matches["team"].astype(str)

players["round"] = players["round"].astype(str)
matches["round"] = matches["round"].astype(str)

players["year"] = players["year"].astype(str)
matches["year"] = matches["year"].astype(str)

players["match_date"] = players["match_date"].astype(str)
matches["match_date"] = matches["match_date"].astype(str)

# Define the merge key
merge_key = [
    "team",
    "round",
    "year",
    "match_date"
]

# Select the match information we need
match_context = matches[merge_key + ["home_away","venue","crowd"]]

# Merge match context into player dataset
enriched_players = players.merge(match_context,on=merge_key,how="left")

In [81]:
# Display the result
print("Enriched dataset:")
print(enriched_players.head())

Enriched dataset:
       id              team  year  career_game_count          opponent round  \
0  556392    Hawthorn Hawks  1994                 17   Richmond Tigers    21   
1  614897      Geelong Cats  2024                  1   St Kilda Saints     1   
2  583553  Essendon Bombers  1999                 97    Adelaide Crows    10   
3  590676  Western Bulldogs  1994                 36   St Kilda Saints    21   
4  582473   Richmond Tigers  1997                113  Melbourne Demons    10   

  result  jersey_num  kicks  marks  ...  bounces  goal_assist  \
0      W          34      5      4  ...        0            0   
1      W           7      5      0  ...        0            0   
2      W           6     14      5  ...        0            0   
3      W          35     12     10  ...        0            0   
4      L          41      4      2  ...        0            0   

   percentage_of_game_played  player_id  match_date  fantasy_points  margin  \
0                        0.0   

In [103]:
print("\nOriginal number of player records:")
print(len(players))

print("\nNumber of records after merge:")
print(len(enriched_players))

print("\nNew columns:")
print(enriched_players[["home_away", "venue", "crowd"]].head())


Original number of player records:
274079

Number of records after merge:
274079

New columns:
  home_away                     venue  crowd
0         A  Melbourne Cricket Ground  52562
1         H             GMHBA Stadium  39352
2         A              AAMI Stadium  39389
3         A         Waverley Park\r\n  14653
4         H  Melbourne Cricket Ground  28879


In [104]:
# Save the enriched dataset
enriched_players.to_csv("Players_Round_By_Round_Enriched.csv",index=False)
print("Enriched dataset saved successfully.")

Enriched dataset saved successfully.


# **Task 3 : Merge Validation**

In [105]:
# Set merge key
merge_key = ["team", "round", "year", "match_date"]

# Select match context
match_context = matches[merge_key + ["home_away", "venue", "crowd"]]

# Check duplicate keys before merging
duplicate_keys = match_context.duplicated(merge_key).sum()
print("Duplicate match keys:", duplicate_keys)

# Merge with indicator
merged = players.merge(match_context,on=merge_key,how="left",indicator=True)

# Find unmatched records
unmatched = merged[merged["_merge"] == "left_only"]
print("\nUnmatched player records:", len(unmatched))

Duplicate match keys: 0

Unmatched player records: 0


In [106]:
# Check record count
original_count = len(players)
merged_count = len(merged)
print("\nOriginal player records:", original_count)
print("Records after merge:", merged_count)
print("Difference:", merged_count - original_count)


Original player records: 274079
Records after merge: 274079
Difference: 0


In [107]:
# Check missing context
print("Missing Home/Away:", merged["home_away"].isna().sum())
print("Missing Venue:", merged["venue"].isna().sum())
print("Missing Crowd:", merged["crowd"].isna().sum())

Missing Home/Away: 0
Missing Venue: 0
Missing Crowd: 0


In [108]:
# Check duplicate rows
duplicate_rows = merged.duplicated().sum()
print("Duplicate rows after merge:", duplicate_rows)

# Remove merge indicator
merged = merged.drop(columns="_merge")

Duplicate rows after merge: 0


In [111]:
# Save final dataset
merged.to_csv("Players_Round_By_Round_Enriched.csv",index=False)

# **Task 4 : Contextual Analysis**

In [112]:
# Load enriched dataset
data = pd.read_csv("Players_Round_By_Round_Enriched.csv")

# Compare home and away performance
home_away = data.groupby("home_away")["fantasy_points"].agg(["count", "mean"]).reset_index()
print("Home vs Away Performance")
print(home_away)

# Calculate home and away averages
home_average = data.loc[data["home_away"] == "H","fantasy_points"].mean()
away_average = data.loc[data["home_away"] == "A","fantasy_points"].mean()
print("\nAverage fantasy points at home:")
print(round(home_average, 2))
print("\nAverage fantasy points away:")
print(round(away_average, 2))
print("\nHome minus Away:")
print(round(home_average - away_average, 2))

Home vs Away Performance
  home_away   count       mean
0         A  137009  64.004408
1         H  137070  66.485504

Average fantasy points at home:
66.49

Average fantasy points away:
64.0

Home minus Away:
2.48


In [113]:
# Check crowd and fantasy points relationship
correlation = data[["crowd", "fantasy_points"]].corr().loc["crowd", "fantasy_points"]
print("\nCrowd and Fantasy Points correlation:")
print(round(correlation, 3))

# Divide crowd into four groups
data["crowd_group"] = pd.qcut(data["crowd"],4,duplicates="drop")
crowd_analysis = data.groupby("crowd_group",observed=True)["fantasy_points"].mean().reset_index()
print("\nAverage Fantasy Points by Crowd Group:")
print(crowd_analysis)


Crowd and Fantasy Points correlation:
0.015

Average Fantasy Points by Crowd Group:
           crowd_group  fantasy_points
0  (1240.999, 23041.0]       65.194984
1   (23041.0, 31946.0]       63.917202
2   (31946.0, 42103.0]       65.739058
3  (42103.0, 101861.0]       66.130090


In [114]:
# Calculate venue performance
venue_performance = data.groupby("venue")["fantasy_points"].agg(["count", "mean"]).reset_index()

# Use venues with at least 100 player records
venue_performance = venue_performance[venue_performance["count"] >= 100]

# Sort from highest to lowest
venue_performance = venue_performance.sort_values("mean",ascending=False)
print("\nTop 10 Venues:")
print(venue_performance.head(10))


Top 10 Venues:
               venue  count       mean
44   TIO Stadium\r\n    112  74.187500
20  Jiangwan Stadium    137  72.481752
49  UTAS Stadium\r\n    133  70.879699
58   Westpac Stadium    134  69.716418
45  TIO Traeger Park    494  69.615385
24      Mars Stadium    639  68.040689
48      UTAS Stadium   4185  67.962007
15     ENGIE Stadium   5010  67.852096
26    Marvel Stadium  50156  67.572354
5      Adelaide Oval  12836  67.563182


# **Task 5 : Data Quality Report**

In [116]:
# Clean team names
players["team"] = players["team"].astype(str).str.strip()
matches["team"] = matches["team"].astype(str).str.strip()

matches["team"] = matches["team"].replace("W. Bulldogs", "Western Bulldogs")

# Set merge key
merge_key = ["team", "round", "year", "match_date"]

# Select context columns
match_context = matches[merge_key + ["home_away", "venue", "crowd"]]

# Check duplicate keys
duplicate_keys = match_context.duplicated(merge_key).sum()

# Merge datasets
merged = players.merge(match_context,on=merge_key,how="left",indicator=True)

# Validation checks
unmatched = (merged["_merge"] == "left_only").sum()
original_records = len(players)
merged_records = len(merged)
extra_records = (merged_records - original_records)

missing_home_away = merged["home_away"].isna().sum()
missing_venue = merged["venue"].isna().sum()
missing_crowd = merged["crowd"].isna().sum()

In [117]:
# Print report
print("DATA QUALITY REPORT")
print()

print("Merge Key:")
print("team + round + year + match_date")

print("\nMerge Strategy:")
print("Left merge")

print("\nDuplicate Match Keys:")
print(duplicate_keys)

print("\nUnmatched Player Records:")
print(unmatched)

print("\nOriginal Player Records:")
print(original_records)

print("\nRecords After Merge:")
print(merged_records)

print("\nExtra Records Created:")
print(extra_records)

print("\nMissing Home/Away:")
print(missing_home_away)

print("\nMissing Venue:")
print(missing_venue)

print("\nMissing Crowd:")
print(missing_crowd)

print("\nData Quality Issue:")
print("W. Bulldogs and Western Bulldogs were standardized.")

print("\nAssumption:")
print("team + round + year + match_date identifies the correct match.")

DATA QUALITY REPORT

Merge Key:
team + round + year + match_date

Merge Strategy:
Left merge

Duplicate Match Keys:
0

Unmatched Player Records:
0

Original Player Records:
274079

Records After Merge:
274079

Extra Records Created:
0

Missing Home/Away:
0

Missing Venue:
0

Missing Crowd:
0

Data Quality Issue:
W. Bulldogs and Western Bulldogs were standardized.

Assumption:
team + round + year + match_date identifies the correct match.
